# Shadow Puppets - Shape Classifier Training

This notebook trains a lightweight CNN for shape classification and exports it as a TFLite model.

## Instructions:
1. Upload your `shapes_training_data.zip` file (generated locally)
2. Run all cells
3. Download the generated `shape_classifier.tflite` and `model_metadata.json`
4. Place them in your local `models/` directory

In [ ]:
# Use TensorFlow 2.16 (default in Colab)
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Upload training data
from google.colab import files
import zipfile
import os

print("Please upload shapes_training_data.zip")
uploaded = files.upload()

# Extract
with zipfile.ZipFile("shapes_training_data.zip", "r") as zip_ref:
    zip_ref.extractall("data")

print("\nExtracted contents:")
for root, dirs, files_list in os.walk("data"):
    for d in dirs:
        path = os.path.join(root, d)
        count = len([f for f in os.listdir(path) if f.endswith('.png')])
        if count > 0:
            print(f"  {d}: {count} images")

In [ ]:
# Configuration
INPUT_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 30
VALIDATION_SPLIT = 0.2

# Find data directory
DATA_DIR = None
for root, dirs, _ in os.walk("data"):
    if any(os.path.isdir(os.path.join(root, d)) and 
           len([f for f in os.listdir(os.path.join(root, d)) if f.endswith('.png')]) > 0 
           for d in dirs):
        DATA_DIR = root
        break

if DATA_DIR is None:
    # Check if shapes are directly in data/
    DATA_DIR = "data"

print(f"Using data directory: {DATA_DIR}")

# List classes
class_names = sorted([d for d in os.listdir(DATA_DIR) 
                      if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith('.')])
print(f"Classes: {class_names}")

In [ ]:
# Load data using Keras preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT,
    # Light augmentation
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(INPUT_SIZE, INPUT_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
)

val_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(INPUT_SIZE, INPUT_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
)

# Get class names in order
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print(f"\nClasses ({num_classes}): {class_names}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")

In [ ]:
# Visualize some training samples
import matplotlib.pyplot as plt
import numpy as np

# Get a batch
images, labels = next(train_generator)

# Plot
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i].squeeze(), cmap='gray')
        label_idx = np.argmax(labels[i])
        ax.set_title(class_names[label_idx])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Build the model - lightweight CNN
from tensorflow.keras import layers, models

def create_model(num_classes, input_size=64):
    model = models.Sequential([
        # Input
        layers.Input(shape=(input_size, input_size, 1)),
        
        # Block 1: 64 -> 32
        layers.Conv2D(32, 3, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2),
        
        # Block 2: 32 -> 16
        layers.Conv2D(64, 3, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2),
        
        # Block 3: 16 -> 8
        layers.Conv2D(128, 3, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2),
        
        # Block 4: 8 -> 4
        layers.Conv2D(128, 3, padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2),
        
        # Classification head
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    
    return model

model = create_model(num_classes, INPUT_SIZE)
model.summary()

In [ ]:
# Compile and train
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=3, monitor='val_loss'),
]

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nFinal validation accuracy: {history.history['val_accuracy'][-1]:.1%}")

In [ ]:
# Evaluate on validation set
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Get predictions
val_generator.reset()
predictions = model.predict(val_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = val_generator.classes

# Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optimize for size and speed
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Ensure compatibility with tflite-runtime 2.14.0
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,  # Standard TFLite ops
]

# Convert
tflite_model = converter.convert()

# Save
tflite_path = "shape_classifier.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"Saved TFLite model: {tflite_path}")
print(f"Model size: {len(tflite_model) / 1024:.1f} KB")

In [ ]:
# Test TFLite model
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:", input_details[0]['shape'])
print("Output shape:", output_details[0]['shape'])

# Test with a sample
val_generator.reset()
test_images, test_labels = next(val_generator)
test_image = test_images[0:1].astype(np.float32)

interpreter.set_tensor(input_details[0]['index'], test_image)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])

pred_idx = np.argmax(output[0])
true_idx = np.argmax(test_labels[0])

print(f"\nTest prediction: {class_names[pred_idx]} ({output[0][pred_idx]:.1%})")
print(f"True label: {class_names[true_idx]}")

In [ ]:
# Save metadata
import json

metadata = {
    "class_names": class_names,
    "input_size": INPUT_SIZE,
    "num_classes": num_classes,
    "model_type": "tflite",
    "validation_accuracy": float(history.history['val_accuracy'][-1]),
}

meta_path = "model_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata: {meta_path}")
print(json.dumps(metadata, indent=2))

In [ ]:
# Download the model files
from google.colab import files

print("Downloading model files...")
files.download("shape_classifier.tflite")
files.download("model_metadata.json")

print("\nDone! Place these files in your local 'models/' directory.")